In [1]:
from tqdm.notebook import tqdm

In [2]:
from transformers import AutoTokenizer

g:\Projects\Visual Studio Code\LMTests\lmtest\lib\site-packages\transformers\utils\hub.py:123: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
model = "Helsinki-NLP/opus-mt-zh-en"
tokenizer = AutoTokenizer.from_pretrained(model)

In [4]:
from datasets import load_dataset

In [5]:
dataset_name = "iwslt2017"

In [6]:
subset = "iwslt2017-en-zh"

In [7]:
iwslt_en_zh = load_dataset(dataset_name, subset)

In [8]:
iwslt_en_zh

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 231266
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 8549
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 879
    })
})

In [9]:
iwslt_train = iwslt_en_zh["train"]

In [10]:
sample = iwslt_train[0]

In [11]:
sample

{'translation': {'en': "Thank you so much, Chris. And it's truly a great honor to have the opportunity to come to this stage twice; I'm extremely grateful.",
  'zh': '非常谢谢，克里斯。的确非常荣幸 能有第二次站在这个台上的机会，我真是非常感激。'}}

In [12]:
tokenizer.encode(sample["translation"]["en"])

[12286,
 24858,
 37,
 192,
 4919,
 17452,
 2,
 7,
 40680,
 5,
 308,
 39,
 21,
 22,
 10096,
 3479,
 466,
 12,
 6374,
 129,
 2757,
 5566,
 800,
 672,
 8,
 53,
 3,
 7,
 6807,
 14327,
 1203,
 3135,
 8,
 731,
 8,
 56,
 7,
 1921,
 5375,
 9224,
 4545,
 10532,
 25,
 26,
 21,
 97,
 7,
 6492,
 59,
 129,
 5904,
 480,
 7,
 18291,
 1552,
 589,
 3479,
 5,
 0]

In [13]:
sample # {'translation': {'en': "Thank you so much, Chris. And it's truly a great honor to have the opportunity to come to this stage twice; I'm extremely grateful.",
  #'zh': '非常谢谢，克里斯。的确非常荣幸 能有第二次站在这个台上的机会，我真是非常感激。'}}

{'translation': {'en': "Thank you so much, Chris. And it's truly a great honor to have the opportunity to come to this stage twice; I'm extremely grateful.",
  'zh': '非常谢谢，克里斯。的确非常荣幸 能有第二次站在这个台上的机会，我真是非常感激。'}}

In [14]:
ens = []
zhs = []

In [15]:
max_len = 99

In [16]:
# tokenizer doesn't have a bos token, so add it manually
bos = "<s>"
tokenizer.add_special_tokens({"bos_token": bos})

1

In [17]:
for sample in tqdm(iwslt_train):
    en = sample["translation"]["en"]
    zh = sample["translation"]["zh"]
    en = tokenizer.encode(en)
    zh = tokenizer.encode(zh)
    if len(en) > max_len or len(zh) > max_len:
        continue
    else:
        # add bos token
        en = [tokenizer.bos_token_id] + en
        zh = [tokenizer.bos_token_id] + zh
        ens.append(en)
        zhs.append(zh)

  0%|          | 0/231266 [00:00<?, ?it/s]

In [18]:
import torch

In [19]:
ens = torch.nested.nested_tensor(ens)
zhs = torch.nested.nested_tensor(zhs)

C:\Users\John\AppData\Local\Temp\ipykernel_14188\3170439811.py:1: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ..\aten\src\ATen\NestedTensorImpl.cpp:180.)
  ens = torch.nested.nested_tensor(ens)


In [20]:
data_dir = "data"
file_name = "processed2.pt"

In [21]:
pad_idx = tokenizer.pad_token_id

In [22]:
ens = torch.nested.to_padded_tensor(ens, pad_idx)

In [23]:
zhs = torch.nested.to_padded_tensor(zhs, pad_idx)

In [24]:
ens.shape

torch.Size([220722, 100])

In [25]:
torch.save((ens, zhs), f"{data_dir}/{file_name}")